# 버전 확인

In [11]:
from pathlib import Path
import importlib
import os

from project_paths import PROJECT_ROOT, DATA_DIR

os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir: {DATA_DIR}')

required_modules = [
    'numpy',
    'pandas',
    'torch',
    'sklearn',
    'pulp',
    'xgboost',
    'socceraction',
    'torch_geometric',
]
for module_name in required_modules:
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, '__version__', 'installed')
        print(f'[OK] {module_name}: {version}')
    except Exception as exc:
        print(f'[MISSING] {module_name}: {exc}')

Project root: /workspace/team-builder
Data dir: /workspace/team-builder/data
[OK] numpy: 1.26.4
[OK] pandas: 2.2.3


/opt/conda/envs/soccer-test/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[OK] torch: 1.13.1+cu117
[OK] sklearn: 1.3.2
[OK] pulp: 2.7.0
[OK] xgboost: 1.7.6
[OK] socceraction: 1.5.3
[OK] torch_geometric: 2.3.1


In [12]:
from pathlib import Path

import pandas as pd

archive_dir = Path('data/archive')
source_files = [
    'matches_European_Championship.csv',
    'matches_France.csv',
    'matches_Germany.csv',
    'matches_Italy.csv',
    'matches_Spain.csv',
    'matches_World_Cup.csv',
]
target_path = archive_dir / 'matches_non_england.csv'

frames = [pd.read_csv(archive_dir / name) for name in source_files]
combined_df = pd.concat(frames, ignore_index=True)

if target_path.exists():
    existing_df = pd.read_csv(target_path)
    if set(combined_df.columns) == set(existing_df.columns):
        combined_df = combined_df[existing_df.columns]

combined_df.to_csv(target_path, index=False)

# spadl 변환

In [ ]:
print('Phase 1: SPADL conversion')
!python -u common_pipeline/convert_wyscout_to_spadl.py

Phase 1: SPADL conversion
[INFO] Loaded players: 3,603
[INFO] Converting England ...
[OK] England: 483,852 actions
[INFO] Converting European_Championship ...
[OK] European_Championship: 62,738 actions
[INFO] Converting France ...
[OK] France: 476,736 actions
[INFO] Converting Germany ...
[OK] Germany: 389,116 actions
[INFO] Converting Italy ...


# vaep 학습

In [ ]:
print('Phase 2: VAEP training and value assignment')
!python -u common_pipeline/train_vaep_xgboost.py

Phase 2: VAEP training and value assignment
[INFO] Selected leagues for training: ['European_Championship', 'France', 'Germany', 'Italy', 'Spain', 'World_Cup']
[INFO] League European_Championship: games=51, actions=62,738
[INFO] League France: games=380, actions=476,736
[INFO] League Germany: games=306, actions=389,116
[INFO] League Italy: games=380, actions=495,820
[INFO] League Spain: games=380, actions=474,641
[INFO] League World_Cup: games=64, actions=80,916
[INFO] Training XGBoost(score)...
[INFO] Training XGBoost(concede)...
[INFO] League England: games=380, actions=483,852
[OK] Saved OOD metrics: /workspace/team-builder/data/vaep/vaep_ood_metrics.csv
[OK] Saved VAEP actions: /workspace/team-builder/data/vaep/vaep_actions.parquet
[OK] Saved models: /workspace/team-builder/data/vaep/vaep_xgb_models.pkl
[OK] Saved holdout metrics: /workspace/team-builder/data/vaep/vaep_holdout_metrics.json
[OK] Rows: 1,979,967


In [ ]:
from pathlib import Path
import pickle
import pandas as pd
import importlib.util
import sys

project_root = Path('/workspace/team-builder')
vaep_dir = project_root / 'data/vaep'
spadl_eng_path = project_root / 'data/spadl/spadl_England.parquet'
out_path = vaep_dir / 'vaep_actions_england_eval.parquet'

print('=== Step 1: VAEP files check ===')
vaep_parquets = sorted([p.name for p in vaep_dir.glob('*.parquet')])
england_keyword_parquets = [n for n in vaep_parquets if 'england' in n.lower()]
print('vaep parquet files:', vaep_parquets)
print('england keyword parquet files:', england_keyword_parquets)

vaep_actions_path = vaep_dir / 'vaep_actions.parquet'
rows_1611_existing = None
if vaep_parquets == ['vaep_actions.parquet'] and vaep_actions_path.exists():
    vaep_actions = pd.read_parquet(vaep_actions_path)
    if 'team_id' in vaep_actions.columns:
        rows_1611_existing = int((pd.to_numeric(vaep_actions['team_id'], errors='coerce') == 1611).sum())
    else:
        rows_1611_existing = 0
    print('rows team_id==1611 in vaep_actions.parquet:', rows_1611_existing)

print('\n=== Step 2: SPADL England check ===')
print('spadl_England exists:', spadl_eng_path.exists())
if not spadl_eng_path.exists():
    raise FileNotFoundError(f'Missing required file: {spadl_eng_path}')

print('\n=== Step 3: Generate England VAEP if missing ===')
need_generate = len(england_keyword_parquets) == 0
print('need_generate:', need_generate)

if need_generate:
    score_model_path = vaep_dir / 'vaep_model_score.pkl'
    concede_model_path = vaep_dir / 'vaep_model_concede.pkl'
    bundle_path = vaep_dir / 'vaep_xgb_models.pkl'

    # Ensure requested split model files exist.
    if (not score_model_path.exists() or not concede_model_path.exists()):
        if not bundle_path.exists():
            raise FileNotFoundError(f'Missing model bundle: {bundle_path}')
        with bundle_path.open('rb') as f:
            payload = pickle.load(f)
        if 'model_scores' not in payload or 'model_concedes' not in payload:
            raise RuntimeError('model bundle missing model_scores/model_concedes')
        if not score_model_path.exists():
            with score_model_path.open('wb') as f:
                pickle.dump(payload['model_scores'], f)
        if not concede_model_path.exists():
            with concede_model_path.open('wb') as f:
                pickle.dump(payload['model_concedes'], f)

    with score_model_path.open('rb') as f:
        model_scores = pickle.load(f)
    with concede_model_path.open('rb') as f:
        model_concedes = pickle.load(f)

    with bundle_path.open('rb') as f:
        payload = pickle.load(f)
    feature_cols = payload.get('feature_cols')
    nb_prev_actions = int(payload.get('nb_prev_actions', 3))
    nr_actions = int(payload.get('nr_actions', 10))
    if not feature_cols:
        raise RuntimeError('feature_cols missing in vaep_xgb_models.pkl')

    phase2_path = project_root / 'common_pipeline/train_vaep_xgboost.py'
    spec = importlib.util.spec_from_file_location('phase2_for_eng_eval', phase2_path)
    phase2_mod = importlib.util.module_from_spec(spec)
    sys.modules['phase2_for_eng_eval'] = phase2_mod
    spec.loader.exec_module(phase2_mod)

    actions = pd.read_parquet(spadl_eng_path)
    vaep_parts = []

    for game_id, game_actions in actions.groupby('game_id', sort=False):
        game_actions = game_actions.sort_values(['action_id', 'time_seconds']).reset_index(drop=True)
        X_game, _ = phase2_mod._compute_features_and_labels_for_game(
            actions=game_actions,
            nb_prev_actions=nb_prev_actions,
            nr_actions=nr_actions,
        )
        X_mat = phase2_mod._align_feature_matrix(X_game[feature_cols].copy(), feature_cols)
        p_scores = pd.Series(model_scores.predict_proba(X_mat)[:, 1])
        p_concedes = pd.Series(model_concedes.predict_proba(X_mat)[:, 1])

        named_actions = phase2_mod.spadl.add_names(game_actions)
        values = phase2_mod.formula.value(named_actions, p_scores, p_concedes).reset_index(drop=True)

        out = pd.concat(
            [
                game_actions.reset_index(drop=True),
                pd.DataFrame({'p_scores': p_scores, 'p_concedes': p_concedes}),
                values,
            ],
            axis=1,
        )
        vaep_parts.append(out)

    if not vaep_parts:
        raise RuntimeError('No England VAEP outputs generated')

    vaep_actions_eng_eval = pd.concat(vaep_parts, ignore_index=True)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    vaep_actions_eng_eval.to_parquet(out_path, index=False)
    print('generated:', out_path)
else:
    print('England parquet already exists; generation skipped')

print('\n=== Step 4: Final report ===')
if out_path.exists():
    final_df = pd.read_parquet(out_path)
    rows_1611 = int((pd.to_numeric(final_df['team_id'], errors='coerce') == 1611).sum()) if 'team_id' in final_df.columns else 0
    print('final_file:', out_path)
    print('rows_total:', len(final_df))
    print('rows_team_id_1611:', rows_1611)
else:
    print('final_file not generated:', out_path)

=== Step 1: VAEP files check ===
vaep parquet files: ['vaep_actions.parquet', 'vaep_actions_england_eval.parquet']
england keyword parquet files: ['vaep_actions_england_eval.parquet']

=== Step 2: SPADL England check ===
spadl_England exists: True

=== Step 3: Generate England VAEP if missing ===
need_generate: False
England parquet already exists; generation skipped

=== Step 4: Final report ===
final_file: /workspace/team-builder/data/vaep/vaep_actions_england_eval.parquet
rows_total: 483852
rows_team_id_1611: 27081


# 전술 탐지 (논문에선 안 쓰임)

In [ ]:
print('Phase 3: tactic detection')
!python -u common_pipeline/detect_tactics_phase3.py

Phase 3: tactic detection
[INFO] England: spadl=483,852, atomic=862,946
[INFO] European_Championship: spadl=62,738, atomic=112,067
[INFO] France: spadl=476,736, atomic=848,964
[INFO] Germany: spadl=389,116, atomic=693,831
[INFO] Italy: spadl=495,820, atomic=885,265
[INFO] Spain: spadl=474,641, atomic=844,039
[INFO] World_Cup: spadl=80,916, atomic=144,758
[OK] Saved atomic actions: /workspace/team-builder/data/tactics/atomic_actions_with_phase.parquet
[OK] Saved phase summary: /workspace/team-builder/data/tactics/phase_summary.parquet
[OK] Saved attack tactic stats: /workspace/team-builder/data/tactics/attack_tactic_stats.csv
[OK] Saved defense tactic stats: /workspace/team-builder/data/tactics/defense_tactic_stats.csv


# vi, io, id 계산 및 팀빌더에 쓰일 람다 계산

In [ ]:
# [Cell 1] 타 리그 데이터로 람다 학습 (OOD 평가를 위함)
!python common_pipeline/compute_synergy_phase4.py \
  --vaep-path data/vaep/vaep_actions.parquet \
  --estimate-lambda \
  --output-dir data/synergy_ood_lambda_train

[OK] Presence built from lineup/substitutions in matches_*.csv
[OK] Estimated lambdas from match results: lambda_vi=0.8728, lambda_io=0.0609, lambda_id=0.0664
[OK] Saved V_I: data/synergy_ood_lambda_train/player_vi.parquet
[OK] Saved I_O: data/synergy_ood_lambda_train/attack_interaction_io.parquet
[OK] Saved I_O event-base: data/synergy_ood_lambda_train/io_event_surfaces_base.parquet
[OK] Saved I_D: data/synergy_ood_lambda_train/defense_interaction_id.parquet
[OK] Saved I_D event-base: data/synergy_ood_lambda_train/id_event_surfaces_base.parquet
[OK] Saved final V: data/synergy_ood_lambda_train/player_synergy_scores.parquet
[OK] players=2,696, io_pairs=35,711, id_pairs=253,916, io_events=2,908,836, id_events=456,075, minutes_rows=3,038


In [ ]:
# [Cell 2] 잉글랜드(맨유) 평가를 위한 시너지 점수판 독립 생성
!python common_pipeline/detect_tactics_phase3.py \
  --vaep-path data/vaep/vaep_actions_england_eval.parquet \
  --output-dir data/tactics_england

!python common_pipeline/compute_synergy_phase4.py \
  --vaep-path data/vaep/vaep_actions_england_eval.parquet \
  --atomic-phase-path data/tactics_england/atomic_actions_with_phase.parquet \
  --matches-csv data/archive/matches_England.csv \
  --output-dir data/synergy_england

[INFO] England: spadl=483,852, atomic=862,946
[INFO] European_Championship: spadl=62,738, atomic=112,067
[INFO] France: spadl=476,736, atomic=848,964
[INFO] Germany: spadl=389,116, atomic=693,831
[INFO] Italy: spadl=495,820, atomic=885,265
[INFO] Spain: spadl=474,641, atomic=844,039
[INFO] World_Cup: spadl=80,916, atomic=144,758
Traceback (most recent call last):
  File "/workspace/team-builder/common_pipeline/detect_tactics_phase3.py", line 422, in <module>
    main()
  File "/workspace/team-builder/common_pipeline/detect_tactics_phase3.py", line 411, in main
    run_phase3(
  File "/workspace/team-builder/common_pipeline/detect_tactics_phase3.py", line 344, in run_phase3
    phase_summary = _build_phase_summary(all_atomic)
  File "/workspace/team-builder/common_pipeline/detect_tactics_phase3.py", line 172, in _build_phase_summary
    g = g.sort_values(["time_seconds", "action_id"]).reset_index(drop=True)
  File "/opt/conda/envs/soccer-test/lib/python3.10/site-packages/pandas/core/fra

# GNN 데이터 셋 생성

In [ ]:
print('Phase 4.5: GNN dataset build (OOD split)')

print('\n[1/2] Build Non-England training graphs')
print('Expected VAEP source: data/vaep/vaep_actions.parquet')
!python -u proposed_spatial_gnn_ga/build_gnn_dataset_phase4_5.py \
  --data-root data \
  --matches-csv data/archive/matches_non_england.csv \
  --output-pt data/phase_4_synergy/data/gnn_phase4_5/hetero_graphs_non_england.pt \
  --output-meta-csv data/phase_4_synergy/data/gnn_phase4_5/hetero_graphs_non_england_meta.csv

print('\n[2/2] Build England evaluation graphs')
print('Expected VAEP source: data/vaep/vaep_actions_england_eval.parquet')
!python -u proposed_spatial_gnn_ga/build_gnn_dataset_phase4_5.py \
  --data-root data \
  --matches-csv data/archive/matches_England.csv \
  --output-pt data/phase_4_synergy/data/gnn_phase4_5/hetero_graphs_england_eval.pt \
  --output-meta-csv data/phase_4_synergy/data/gnn_phase4_5/hetero_graphs_england_eval_meta.csv

Phase 4.5: GNN dataset build (OOD split)

[1/2] Build Non-England training graphs
Expected VAEP source: data/vaep/vaep_actions.parquet
[INFO] selected VAEP: data/vaep/vaep_actions.parquet
[INFO] selected IO:   data/synergy/io_event_surfaces_base.parquet
[INFO] selected ID:   data/synergy/id_event_surfaces_base.parquet
[OK] saved graphs: data/phase_4_synergy/data/gnn_phase4_5/hetero_graphs_non_england.pt
[OK] saved meta:   data/phase_4_synergy/data/gnn_phase4_5/hetero_graphs_non_england_meta.csv
[OK] n_graphs:     1561
 match_id                match_time  n_past_matches  n_off_events  n_def_events  n_io_events  n_id_events
  1694390 2016-06-10 19:00:00+00:00               0             0             0            0            0
  1694391 2016-06-11 13:00:00+00:00               1          1234          1234         1274          272
  1694396 2016-06-11 16:00:00+00:00               2          2488          2488         2902          468
  1694397 2016-06-11 19:00:00+00:00               3 

# GNN 학습 (비잉글랜드로)

In [ ]:
print('Phase 5: GNN training (2-layer anti-oversmoothing)')
!python -u proposed_spatial_gnn_ga/train_gnn_phase5.py \
  --epochs 50 \
  --num-layers 3 \
  --output-model data/phase_5_lineup/data/gnn_phase5/hetero_edge_gat_win_epoch10_l2.pt

Phase 5: GNN training (2-layer anti-oversmoothing)
[INFO] fitted scaler on train fold and transformed train/valid graphs
/opt/conda/envs/soccer-test/lib/python3.10/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
[Epoch 001] train_loss=1.07571 val_loss=1.05881 val_acc=0.4391 val_macro_f1=0.2034
[Epoch 002] train_loss=1.03867 val_loss=1.04894 val_acc=0.4423 val_macro_f1=0.2114
[Epoch 003] train_loss=1.01122 val_loss=1.00428 val_acc=0.5192 val_macro_f1=0.3930
[Epoch 004] train_loss=0.99826 val_loss=0.99603 val_acc=0.5256 val_macro_f1=0.3971
[Epoch 005] train_loss=0.98440 val_loss=0.99671 val_acc=0.5256 val_macro_f1=0.3946
[Epoch 006] train_loss=0.96767 val_loss=1.02891 val_acc=0.5128 val_macro_f1=0.3851
[Epoch 007] train_loss=0.94887 val_loss=1.03642 val_acc=0.5000 val_macro_f1=0.3666
[Epo

# 학습된 GNN + GA 를 통한 라인업 추출

In [ ]:
!python -u proposed_spatial_gnn_ga/optimze_lineup_ga_phase6.py

# epl 380 경기 

In [ ]:
print('EPL 2017-18 전체(380경기) OOD 비교 평가를 실행합니다.')
print('Leakage 차단 설정: league_mode=england + tactics_england/synergy_england/lambda_weights.csv 경로만 사용')

full_output_dir = 'data/phase_6_validation/data/batch_england_full_season_ood'

!python -u evaluation_comparison/run_experiment_batch.py \
  --league-mode england \
  --matches-csv data/archive/matches_England.csv \
  --history-matches-csv data/archive/matches_England.csv \
  --max-matches 0 \
  --model-ckpt data/phase_5_lineup/data/gnn_phase5/hetero_edge_gat_win_ood_final.pt \
  --tb-tactics-dir data/tactics_england \
  --tb-synergy-dir data/synergy_england \
  --tb-lambda-csv data/synergy_ood_lambda_train/lambda_weights.csv \
  --tb-lambda-scaler-stats data/synergy_ood_lambda_train/lambda_scaler_stats.csv \
  --output-dir data/phase_6_validation/data/batch_england_full_season_ood

from pathlib import Path
import pandas as pd

summary_path = Path(full_output_dir) / 'hitrate_eval' / 'hitrate_summary.csv'
if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    print('\n=== Final Hit-Rate / Jaccard Summary (EPL 380) ===')
    print(summary_df.to_string(index=False))
else:
    print(f'[WARN] summary 파일이 없습니다: {summary_path}')

EPL 2017-18 전체(380경기) OOD 비교 평가를 실행합니다.
Leakage 차단 설정: league_mode=england + tactics_england/synergy_england/lambda_weights.csv 경로만 사용
[INFO] loading GNN model on device=cuda
/opt/conda/envs/soccer-test/lib/python3.10/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
[INFO] using feature scaler from model checkpoint
[INFO] start batch: matches=380 (pred rows target=760)
[INFO] loaded player synergy scores: /workspace/team-builder/data/synergy_england/player_synergy_scores.parquet rows=514
[OK] team_id=1609, formation=4-4-2, status=Optimal
[OK] objective_total=2.778920 (V_I=2.519828, I_O=0.208217, I_D=0.050875)
[OK] saved: data/phase_6_validation/data/batch_england_full_season_ood/_tmp_teambuilder_home/lineup_selected.parquet
[INFO] loaded player synergy scores: /workspace/team-builder/dat

In [ ]:
print('맨유(Home) vs Stoke City / Manchester City 타겟 분석을 실행합니다.')
print('Leakage 차단 설정: league_mode=england + tactics_england/synergy_england/lambda_weights.csv 경로만 사용')

!python -u evaluation_comparison/run_experiment_batch.py \
  --league-mode england \
  --matches-csv data/archive/matches_England.csv \
  --history-matches-csv data/archive/matches_England.csv \
  --model-ckpt data/phase_5_lineup/data/gnn_phase5/hetero_edge_gat_win_ddp_full.pt \
  --tb-tactics-dir data/tactics_england \
  --tb-synergy-dir data/synergy_england \
  --tb-lambda-csv data/synergy_ood_lambda_train/lambda_weights.csv \
  --tb-lambda-scaler-stats data/synergy_ood_lambda_train/lambda_scaler_stats.csv \
  --filter-home-team "Manchester City" \
  --filter-away-team "Stoke City" \
  --output-dir data/phase_6_validation/data/mu_vs_stoke_ood \
  --skip-evaluate

!python -u evaluation_comparison/run_experiment_batch.py \
  --league-mode england \
  --matches-csv data/archive/matches_England.csv \
  --history-matches-csv data/archive/matches_England.csv \
  --model-ckpt data/phase_5_lineup/data/gnn_phase5/hetero_edge_gat_win_ood_final.pt \
  --tb-tactics-dir data/tactics_england \
  --tb-synergy-dir data/synergy_england \
  --tb-lambda-csv data/synergy_ood_lambda_train/lambda_weights.csv \
  --tb-lambda-scaler-stats data/synergy_ood_lambda_train/lambda_scaler_stats.csv \
  --filter-home-team "Manchester United" \
  --filter-away-team "Manchester City" \
  --output-dir data/phase_6_validation/data/mu_vs_mancity_ood \
  --skip-evaluate

import ast
from pathlib import Path
import pandas as pd

players = pd.read_csv('data/archive/players.csv')
teams = pd.read_csv('data/archive/teams.csv')
matches = pd.read_csv('data/archive/matches_England.csv')

pid_series = pd.to_numeric(players['wyId'], errors='coerce')
name_col = 'shortName' if 'shortName' in players.columns else 'name'
id_to_name = {int(pid): str(name) for pid, name in zip(pid_series, players[name_col]) if pd.notna(pid)}

team_ids = {str(n): int(i) for n, i in zip(teams['name'], pd.to_numeric(teams['wyId'], errors='coerce')) if pd.notna(i)}
mu_id = team_ids['Manchester United']

def _safe_eval_list(v):
    if isinstance(v, list):
        return v
    if isinstance(v, str):
        t = v.strip()
        if not t:
            return []
        try:
            obj = ast.literal_eval(t)
        except Exception:
            return []
        return obj if isinstance(obj, list) else []
    return []

def _parse_lineup_ids(raw):
    out = []
    for row in _safe_eval_list(raw):
        if isinstance(row, dict):
            pid = row.get('playerId')
        else:
            pid = row
        try:
            pid = int(float(pid))
        except Exception:
            continue
        if pid not in out:
            out.append(pid)
        if len(out) >= 11:
            break
    return out

def _lineup_ids_to_names(ids):
    return [id_to_name.get(int(pid), f'UNKNOWN({int(pid)})') for pid in ids]

def _extract_mu_compare(output_dir: str, away_name: str) -> pd.DataFrame:
    out_dir = Path(output_dir)
    gnn = pd.read_csv(out_dir / 'gnn_predictions.csv')
    tb = pd.read_csv(out_dir / 'teambuilder_predictions.csv')

    gnn_mu = gnn[(pd.to_numeric(gnn['team_id'], errors='coerce') == mu_id)].copy()
    tb_mu = tb[(pd.to_numeric(tb['team_id'], errors='coerce') == mu_id)].copy()

    if 'opponent_team_name' in gnn_mu.columns:
        gnn_mu = gnn_mu[gnn_mu['opponent_team_name'] == away_name].copy()
    if 'opponent_team_name' in tb_mu.columns:
        tb_mu = tb_mu[tb_mu['opponent_team_name'] == away_name].copy()

    if gnn_mu.empty and tb_mu.empty:
        raise RuntimeError(f'No Manchester United rows found for away={away_name} in {output_dir}')

    if not gnn_mu.empty and 'match_time' in gnn_mu.columns:
        gnn_mu = gnn_mu.sort_values('match_time')
    if not tb_mu.empty and 'match_time' in tb_mu.columns:
        tb_mu = tb_mu.sort_values('match_time')

    if not gnn_mu.empty:
        game_id = int(gnn_mu.iloc[-1]['game_id'])
    else:
        game_id = int(tb_mu.iloc[-1]['game_id'])

    m = matches[pd.to_numeric(matches['wyId'], errors='coerce') == game_id]
    if m.empty:
        raise RuntimeError(f'match not found in matches_England.csv: {game_id}')
    mr = m.iloc[0]

    if int(float(mr['team1.teamId'])) == mu_id:
        actual_ids = _parse_lineup_ids(mr['team1.formation.lineup'])
    else:
        actual_ids = _parse_lineup_ids(mr['team2.formation.lineup'])

    gnn_row = gnn_mu[pd.to_numeric(gnn_mu['game_id'], errors='coerce') == game_id]
    tb_row = tb_mu[pd.to_numeric(tb_mu['game_id'], errors='coerce') == game_id]

    gnn_ids = [int(x) for x in str(gnn_row.iloc[-1]['lineup_ids']).split('|') if str(x).strip()] if not gnn_row.empty else []
    tb_ids = [int(x) for x in str(tb_row.iloc[-1]['lineup_ids']).split('|') if str(x).strip()] if not tb_row.empty else []

    match_label = f"Manchester United vs {away_name} (game_id={game_id})"
    rows = [
        {'Match': match_label, 'Model': 'Actual Coach XI', 'Starting XI (Names)': ', '.join(_lineup_ids_to_names(actual_ids))},
        {'Match': match_label, 'Model': 'GNN XI', 'Starting XI (Names)': ', '.join(_lineup_ids_to_names(gnn_ids))},
        {'Match': match_label, 'Model': 'Team-Builder XI', 'Starting XI (Names)': ', '.join(_lineup_ids_to_names(tb_ids))},
    ]
    return pd.DataFrame(rows)

cmp_stoke = _extract_mu_compare('data/phase_6_validation/data/mu_vs_stoke_ood', 'Stoke City')
cmp_mancity = _extract_mu_compare('data/phase_6_validation/data/mu_vs_mancity_ood', 'Manchester City')
compare_long = pd.concat([cmp_stoke, cmp_mancity], ignore_index=True)

# 이름 잘림 방지 옵션
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)

print('\n=== Manchester United Target Match Lineup Comparison (Long) ===')
display(compare_long)

compare_wide = compare_long.pivot(index='Match', columns='Model', values='Starting XI (Names)').reset_index()
print('\n=== Manchester United Target Match Lineup Comparison (Wide) ===')
display(compare_wide)

# 모델별 11명을 줄바꿈 + 번호로 전체 출력(절대 잘리지 않음)
print('\n=== Full XI Names (No Truncation) ===')
for _, row in compare_long.iterrows():
    names = [x.strip() for x in str(row['Starting XI (Names)']).split(',') if x.strip()]
    print(f"\n[{row['Match']}] {row['Model']}")
    for i, name in enumerate(names, start=1):
        print(f"{i:02d}. {name}")

맨유(Home) vs Stoke City / Manchester City 타겟 분석을 실행합니다.
Leakage 차단 설정: league_mode=england + tactics_england/synergy_england/lambda_weights.csv 경로만 사용
[INFO] filtered to home team 'Manchester United' (1611): 20 matches
[INFO] filtered to away team 'Stoke City' (1639): 2 matches
[INFO] loading GNN model on device=cuda
/opt/conda/envs/soccer-test/lib/python3.10/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
[INFO] using feature scaler from model checkpoint
[INFO] start batch: matches=2 (pred rows target=4)
[INFO] loaded player synergy scores: /workspace/team-builder/data/synergy_england/player_synergy_scores.parquet rows=514
[OK] team_id=1611, formation=4-4-2, status=Optimal
[OK] objective_total=2.895321 (V_I=2.683383, I_O=0.203736, I_D=0.008203)
[OK] saved: data/phase_6_validation/data/m

,Match,Model,Starting XI (Names)
0,Manchester United vs Stoke City (game_id=2499944),Actual Coach XI,"A. Martial, J. Lingard, Juan Mata, David de Gea, P. Jones, A. Valencia, P. Pogba, C. Smalling, R. Lukaku, L. Shaw, N. Mati\u0107"
1,Manchester United vs Stoke City (game_id=2499944),GNN XI,"David de Gea, N. Mati\u0107, Juan Mata, M. Rashford, P. Jones, M. Rojo, P. Pogba, J. Lingard, A. Martial, R. Lukaku, M. Fellaini"
2,Manchester United vs Stoke City (game_id=2499944),Team-Builder XI,"V. Lindel\u00f6f, A. Valencia, C. Smalling, P. Jones, A. Martial, M. Rashford, David de Gea, J. Lingard, P. Pogba, M. Fellaini, Juan Mata"
3,Manchester United vs Manchester City (game_id=2500045),Actual Coach XI,"A. S\u00e1nchez, J. Lingard, David de Gea, A. Valencia, P. Pogba, C. Smalling, R. Lukaku, N. Mati\u0107, E. Bailly, A. Young, Ander Herrera"
4,Manchester United vs Manchester City (game_id=2500045),GNN XI,"David de Gea, A. Young, A. Martial, V. Lindel\u00f6f, R. Lukaku, P. Pogba, N. Mati\u0107, A. S\u00e1nchez, M. Rojo, J. Lingard, M. Rashford"
5,Manchester United vs Manchester City (game_id=2500045),Team-Builder XI,"V. Lindel\u00f6f, A. Valencia, A. Young, C. Smalling, R. Lukaku, A. Martial, David de Gea, J. Lingard, P. Pogba, Juan Mata, N. Mati\u0107"



=== Manchester United Target Match Lineup Comparison (Wide) ===


Model,Match,Actual Coach XI,GNN XI,Team-Builder XI
0,Manchester United vs Manchester City (game_id=2500045),"A. S\u00e1nchez, J. Lingard, David de Gea, A. Valencia, P. Pogba, C. Smalling, R. Lukaku, N. Mati\u0107, E. Bailly, A. Young, Ander Herrera","David de Gea, A. Young, A. Martial, V. Lindel\u00f6f, R. Lukaku, P. Pogba, N. Mati\u0107, A. S\u00e1nchez, M. Rojo, J. Lingard, M. Rashford","V. Lindel\u00f6f, A. Valencia, A. Young, C. Smalling, R. Lukaku, A. Martial, David de Gea, J. Lingard, P. Pogba, Juan Mata, N. Mati\u0107"
1,Manchester United vs Stoke City (game_id=2499944),"A. Martial, J. Lingard, Juan Mata, David de Gea, P. Jones, A. Valencia, P. Pogba, C. Smalling, R. Lukaku, L. Shaw, N. Mati\u0107","David de Gea, N. Mati\u0107, Juan Mata, M. Rashford, P. Jones, M. Rojo, P. Pogba, J. Lingard, A. Martial, R. Lukaku, M. Fellaini","V. Lindel\u00f6f, A. Valencia, C. Smalling, P. Jones, A. Martial, M. Rashford, David de Gea, J. Lingard, P. Pogba, M. Fellaini, Juan Mata"



=== Full XI Names (No Truncation) ===

[Manchester United vs Stoke City (game_id=2499944)] Actual Coach XI
01. A. Martial
02. J. Lingard
03. Juan Mata
04. David de Gea
05. P. Jones
06. A. Valencia
07. P. Pogba
08. C. Smalling
09. R. Lukaku
10. L. Shaw
11. N. Mati\u0107

[Manchester United vs Stoke City (game_id=2499944)] GNN XI
01. David de Gea
02. N. Mati\u0107
03. Juan Mata
04. M. Rashford
05. P. Jones
06. M. Rojo
07. P. Pogba
08. J. Lingard
09. A. Martial
10. R. Lukaku
11. M. Fellaini

[Manchester United vs Stoke City (game_id=2499944)] Team-Builder XI
01. V. Lindel\u00f6f
02. A. Valencia
03. C. Smalling
04. P. Jones
05. A. Martial
06. M. Rashford
07. David de Gea
08. J. Lingard
09. P. Pogba
10. M. Fellaini
11. Juan Mata

[Manchester United vs Manchester City (game_id=2500045)] Actual Coach XI
01. A. S\u00e1nchez
02. J. Lingard
03. David de Gea
04. A. Valencia
05. P. Pogba
06. C. Smalling
07. R. Lukaku
08. N. Mati\u0107
09. E. Bailly
10. A. Young
11. Ander Herrera

[Manchester Unit